In [6]:
# -----------------------------
# Week 12: Product Recommendation System
# -----------------------------

# Import libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler
from tabulate import tabulate

# -----------------------------
# STEP 1: Load Final Features from Week 11
# -----------------------------
# Assuming Week 11 output is saved as 'final_features.csv'
df = pd.read_csv("Customer_support_data.csv")

print("Week 12: Loaded Final Features")
print(tabulate(df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 2: Prepare User-Item Matrix
# -----------------------------
# Create pivot table: rows=CustomerID, columns=Product_category, values=CSAT Score
user_item_matrix = df.pivot_table(
    index='Unique id',
    columns='Product_category',
    values='CSAT Score',
    fill_value=0
)

print("User-Item Matrix Sample:")
print(tabulate(user_item_matrix.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 3: Compute Item Similarity (Cosine Similarity)
# -----------------------------
item_similarity = cosine_similarity(user_item_matrix.T)  # transpose to get items as rows
item_similarity_df = pd.DataFrame(item_similarity, index=user_item_matrix.columns, columns=user_item_matrix.columns)

print("Item Similarity Matrix Sample:")
print(tabulate(item_similarity_df.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 4: Predict Ratings for All Users
# -----------------------------
predicted_ratings = user_item_matrix.dot(item_similarity_df) / item_similarity_df.sum(axis=1)
predicted_ratings = predicted_ratings.fillna(0)

print("Predicted Ratings Sample:")
print(tabulate(predicted_ratings.head(), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 5: Generate Top-N Recommendations
# -----------------------------
def top_n_recommendations(pred_ratings, n=3):
    recommendations = {}
    for user in pred_ratings.index:
        # Sort products by predicted score
        top_products = pred_ratings.loc[user].sort_values(ascending=False).head(n).index.tolist()
        recommendations[user] = top_products
    return recommendations

top_recs = top_n_recommendations(predicted_ratings, n=3)

# Convert to DataFrame for table display
top_recs_df = pd.DataFrame(list(top_recs.items()), columns=['UserID', 'Top_Products'])

print("Top 3 Recommended Products per User:")
print(tabulate(top_recs_df.head(10), headers='keys', tablefmt='fancy_grid'), "\n")

# -----------------------------
# STEP 6: Optional - Save Recommendations
# -----------------------------
top_recs_df.to_csv("week12_recommendations.csv", index=False)
print("Top recommendations saved to 'week12_recommendations.csv'")


Week 12: Loaded Final Features
╒════╤══════════════════════════════════════╤════════════════╤═════════════════╤══════════════════════════════╤════════════════════╤══════════════════════════════════════╤═══════════════════╤═════════════════════╤═══════════════════╤════════════════════════╤═════════════════╤════════════════════╤══════════════╤═══════════════════════════╤═════════════════════╤════════════════╤═════════════════╤═════════════════╤═══════════════╤══════════════╕
│    │ Unique id                            │ channel_name   │ category        │ Sub-category                 │   Customer Remarks │ Order_id                             │   order_date_time │ Issue_reported at   │ issue_responded   │ Survey_response_Date   │   Customer_City │   Product_category │   Item_price │   connected_handling_time │ Agent_name          │ Supervisor     │ Manager         │ Tenure Bucket   │ Agent Shift   │   CSAT Score │
╞════╪══════════════════════════════════════╪════════════════╪═════════════